In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import acf, pacf, adfuller

from scipy.stats import pearsonr, spearmanr

import json
import sys
from pathlib import Path

In [ ]:
sys.path.append(str(Path.cwd().parent))

from src.config import (
    file_names,
    target,
    predictors,
    variables,
    numerical_cols,
    sentinel_col
)

In [ ]:
for file in file_names:
    print(f"\n{'='*25}")
    print(f"FILE: {file}")
    print(f"{'='*25}")

    df_check = pd.read_csv(f"../data/{file}")

    print("Shape:", df_check.shape)
    print("\nColumns:", df_check.columns.tolist())
    print("\nThe first 5 rows:\n", df_check.head())
    print("\nData type:\n", df_check.dtypes)
    print("\nMissing values:\n", df_check.isnull().sum())
    print("\nDuplicates rows:\n", df_check.duplicated().sum())

In [ ]:
df = pd.read_csv("../data/combined_gee_pm25.csv")

df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [ ]:
print("\nMain dataset info:")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Target:", target)
print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nLatitude range:")
print(df["latitude"].min(), df["latitude"].max())

print("\nLongitude range:")
print(df["longitude"].min(), df["longitude"].max())

print("\nDataset information:")
df.info()

print("\nDescribe:\n", df[target].describe())

In [ ]:
summary = df[numerical_cols].describe().T

summary["skewness"] = df[numerical_cols].skew()
summary["kurtosis"] = df[numerical_cols].kurt()

print("\nDescribe:\n", summary.round(2))

In [ ]:
print("\nObservation per sensor:")
print(df["sensor_name"].value_counts())

print("\nNumber of unique sensors:")
print(df["sensor_name"].nunique())

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(variables):
    sns.histplot(data=df, x=col, bins=40, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(variables):
    sns.boxenplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
top_sensors = df["sensor_name"].value_counts().head(10).index

plt.figure(figsize=(12, 6))

sns.countplot(
    data=df[df["sensor_name"].isin(top_sensors)],
    y="sensor_name",
    order=top_sensors
)

plt.title("Top 10 sensors by number of observations")
plt.xlabel("Number of observations")
plt.ylabel("sensor")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))

sns.boxplot(
    data=df[df["sensor_name"].isin(top_sensors)],
    x="sensor_name",
    y=target,
    order=top_sensors
)

plt.title("PM2.5 Distributions")
plt.xlabel("sensor")
plt.ylabel("PM2.5")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
print("\nMissing count:\n", df.isnull().sum())

print(
    "\nMissing percentage:\n",
    (df.isnull().mean() * 100).round(2)
)

In [ ]:
print("\nDomain knowledge check:")

print("Negative PM2.5 values:", (df[target] < 0).sum())
print("PM2.5 values equal to -999:", (df[target] == -999).sum())
print("PM2.5 values equal to 1000:", (df[target] == 1000).sum())

print("Negative AOD values:", (df["AOD"] < 0).sum())
print(
    "Negative pressure values:",
    (df["pressure_mb"] < 0).sum()
)

In [ ]:
print("\nIQR:")

iqr_results = []

for col in variables:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = (
        (df[col] < lower) |
        (df[col] > upper)
    ).sum()

    iqr_results.append({
        "variable": col,
        "Lower bound": lower,
        "Upper bound": upper,
        "Outliers": outliers,
        "Percentage": outliers / len(df) * 100
    })

iqr_table = pd.DataFrame(iqr_results)

print(iqr_table.round(2).to_string(index=False))

In [ ]:
print("\nZ-SCORES:")

zscore = (
    df[variables] - df[variables].mean()
) / df[variables].std()

zscore_results = pd.DataFrame({
    "Variable": variables,
    "Outliers": (zscore.abs() > 3).sum().values
})

zscore_results["Percentage"] = (
    zscore_results["Outliers"] / len(df) * 100
)

print(zscore_results.round(2).to_string(index=False))

In [ ]:
print("\nIsolation forest:")

X_if = df[predictors].copy()

imputer = SimpleImputer(strategy="median")
X_if_imputed = imputer.fit_transform(X_if)

if_scaler = StandardScaler()
X_if_scaled = if_scaler.fit_transform(X_if_imputed)

iso_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

if_predictions = iso_forest.fit_predict(X_if_scaled)

n_out_if = (if_predictions == -1).sum()
pct_out_if = n_out_if / len(df) * 100

print("Potential outliers:", n_out_if)
print("Normal observations:", (if_predictions == 1).sum())
print("Outliers percentage:", round(pct_out_if, 2), "%")

df["isolation_outlier"] = if_predictions

print("\nFirst 10 potential outliers:")

print(
    df[df["isolation_outlier"] == -1][
        ["date", "sensor_name", "pm25", "AOD", "NO2", "CO", "O3"]
    ].head(10)
)

In [ ]:
X = df[predictors]

pearson_corr = X.corr(method="pearson")
spearman_corr = X.corr(method="spearman")

print("Pearson correlation with PM2.5:")
print(
    df[predictors + [target]]
    .corr()[target]
    .sort_values(ascending=False)
)

print("\nSpearman correlation with PM2.5:")
print(
    df[predictors + [target]]
    .corr(method="spearman")[target]
    .sort_values(ascending=False)
)

In [ ]:
upper_triangle = pd.DataFrame(
    np.triu(
        np.ones(pearson_corr.shape),
        k=1
    ).astype(bool),
    index=pearson_corr.index,
    columns=pearson_corr.columns
)

high_corr = pearson_corr.where(upper_triangle).stack()

print("\nHighly correlated predictors:")
print(high_corr[high_corr.abs() > 0.9])

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    pearson_corr,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Pearson correlation heatmap")
plt.tight_layout()
plt.show()


plt.figure(figsize=(12, 8))

sns.heatmap(
    spearman_corr,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Spearman correlation heatmap")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(predictors):
    sns.scatterplot(
        data=df,
        x=col,
        y=target,
        alpha=0.3,
        ax=axes[i]
    )

    axes[i].set_title(f"PM2.5 vs {col}")

plt.tight_layout()
plt.show()

In [ ]:
df["month"] = df["date"].dt.month

seasons = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

df["seasons"] = df["month"].map(seasons)

plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="seasons",
    y=target,
    order=["Winter", "Spring", "Summer", "Autumn"]
)

plt.title("PM2.5 distribution by season")
plt.show()

print("\nAverage PM2.5 by seasons:")
print(df.groupby("seasons")[target].mean().round(2))

In [ ]:
print(
    "PM2.5 skewness:",
    round(df[target].skew(), 2)
)

sns.histplot(
    data=df,
    x=target,
    bins=40,
    kde=True
)

plt.title("PM2.5 target distribution")
plt.xlabel("PM2.5 (µg/m³)")
plt.show()

In [ ]:
prop_25 = (df[target] > 25).mean() * 100
prop_50 = (df[target] > 50).mean() * 100

print("\nPM2.5 pollution thresholds:")
print(
    "Proportion of PM2.5 > 25:",
    round(prop_25, 2),
    "%"
)

print(
    "Proportion of PM2.5 > 50:",
    round(prop_50, 2),
    "%"
)

In [ ]:
X_lof = df[predictors].copy()

lof_imputer = SimpleImputer(
    strategy="median"
)

X_lof_imputed = lof_imputer.fit_transform(
    X_lof
)

lof_scaler = StandardScaler()

X_lof_scaled = lof_scaler.fit_transform(
    X_lof_imputed
)

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination="auto",
    n_jobs=-1
)

lof_predictions = lof.fit_predict(
    X_lof_scaled
)

n_out_lof = (
    lof_predictions == -1
).sum()

pct_out_lof = (
    n_out_lof / len(df) * 100
)

print("LOF potential outliers:", n_out_lof)
print(
    "LOF outlier percentage:",
    round(pct_out_lof, 2),
    "%"
)

In [ ]:
X_vif = df[predictors].copy()

X_vif = SimpleImputer(
    strategy="median"
).fit_transform(X_vif)

X_vif = pd.DataFrame(
    StandardScaler().fit_transform(X_vif),
    columns=predictors
)

vif_results = {}

for i, predictor in enumerate(predictors):

    try:
        vif_results[predictor] = (
            variance_inflation_factor(
                X_vif.values,
                i
            )
        )

    except (ValueError, np.linalg.LinAlgError):
        vif_results[predictor] = np.nan

print("\nVIF results:")

for predictor, value in vif_results.items():
    print(
        predictor,
        ":",
        round(value, 3)
        if pd.notna(value)
        else np.nan
    )